# Process a GRS L2A Zarr pyramid

Recent GRS versions write the Level-2A product as a **multiscale Zarr store**: the image
is stored at several resolutions (pyramid levels), level `0` being the full resolution
and each next level 2 times coarser. This tutorial:

1. opens a GRS L2A Zarr pyramid and lists its levels;
2. makes a **quick look** of the main L2B products from a coarse level, in a few seconds;
3. processes the **full resolution** and writes the L2B product to a file;
4. gives the equivalent command line.

The output variables are described in {doc}`../output_product`; the
{doc}`grsl2bgen_l2b_full_process` tutorial explores them in detail.

In [ ]:
import logging
import os
import shutil
import tempfile

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import GRSl2bgen

logging.getLogger().setLevel(logging.WARNING)   # logging.INFO to follow the processing

# lighter figures for the documentation
%config InlineBackend.figure_formats = ['jpeg']
plt.rcParams['figure.dpi'] = 72

print(f'GRSl2bgen {GRSl2bgen.__version__}')

## The L2A Zarr pyramid

`l2a_path` is a GRS L2A product folder holding `<name>.zarr` (a path to the `.zarr`
store itself works too). Replace it with the path of your own product. The store holds
one group per level (`0`, `1`, ...) plus an `ancillary` group, and its root attribute
`multiscales` lists the levels:

In [ ]:
l2a_path = '/data/satellite/Sentinel-2/test/S2A_MSIL2AGRS_20170412T110621_N0500_R137_T30TWT_20231111T110836_V3.0.1dev'
store = os.path.join(l2a_path, os.path.basename(l2a_path) + '.zarr')

layout = xr.open_zarr(store).attrs['multiscales'][0]['layout']
pixel = abs(float(xr.open_zarr(store, group='0').x.diff('x')[0]))
levels = pd.DataFrame([{'level': k, 'group': lev['group'], 'pixels (y x x)': ' x '.join(map(str, lev['shape'])),
                        'pixel size (m)': pixel * lev['scale']} for k, lev in enumerate(layout)]).set_index('level')
levels

## Quick look at a coarse level

Processing time and memory scale with the number of pixels: level `k` has $4^k$ times
fewer pixels than the full resolution. A quick-look panel is only a few hundred pixels
wide on screen, so there is no gain in processing more than ~1000 pixels per side; below
that size the run time is dominated by the fixed cost of setting up the processing chain
(a few seconds), so there is no gain either in going coarser. The quick look therefore
takes the **first level whose larger side is at most 1000 pixels** (level 3, 160 m, for
a Sentinel-2 tile: about 2 times faster than level 2 and 60 times fewer pixels than the
full resolution).

`level` is passed to {py:class}`~GRSl2bgen.process.Process` (or `--level` on the command
line); `execute()` builds the L2B product lazily in memory, without writing it, and
`compute()` loads it.

In [ ]:
%%time
level = next(k for k, lev in enumerate(layout) if max(lev['shape']) <= 1000)
print(f'quick-look level: {level}, {levels.loc[level, "pixels (y x x)"]} pixels of '
      f'{levels.loc[level, "pixel size (m)"]:.0f} m')

proc = GRSl2bgen.Process(l2a_path, level=level)
proc.execute()
l2b = proc.l2b.l2b_prod.compute()

Maps of the main products, with the true-colour composite of the L2A image. The OWT
maps show the best-matching class. Chlorophyll-a, SPM, turbidity and the detritus and
CDOM absorption `adg443_QAAv6Chl` are shown on fixed logarithmic scales, as they vary
over orders of magnitude; the other colour scales are clipped to the 2-98 % percentiles
(`robust=True`).

In [ ]:
def show(ax, da, title, **kwargs):
    """Map one variable with equal aspect and no axes."""
    da.plot.imshow(ax=ax, cbar_kwargs={'shrink': 0.8, 'label': da.attrs.get('units', '')}, **kwargs)
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_axis_off()

# colormap for the CDOM and detritus absorption
colors = ["#E0F7FA","royalblue", "gold", "#3E2723"]
cdom_cmap = mpl.colors.LinearSegmentedColormap.from_list("DeepBlueToTea", colors, N=256)

rrs = proc.l2a.raster.Rrs
date = str(l2b.time.dt.strftime('%Y-%m-%d').values)
maps = [('owt_index_Spyrakos2018', dict(cmap='tab20', vmin=0.5, vmax=13.5)),
        ('Chla_OWTblend_Ta2025', dict(cmap='Spectral_r', norm=mpl.colors.LogNorm(0.3, 30))),
        ('SPM_OWTblend_Ta2025', dict(cmap='YlOrBr', norm=mpl.colors.LogNorm(0.1, 300))),
        ('TURB_dogliotti', dict(cmap='YlOrBr', norm=mpl.colors.LogNorm(0.1, 300))),
        ('acdom_B15', dict(cmap=cdom_cmap, robust=True)),
        ('adg443_QAAv6Chl', dict(cmap=cdom_cmap, norm=mpl.colors.LogNorm(0.01, 3))),
        ('Kd_par', dict(cmap='magma_r', robust=True)),
        ('Zsd_Lee15', dict(cmap='Blues', robust=True))]

fig, axs = plt.subplots(3, 3, figsize=(16, 14), constrained_layout=True)
rrs.sel(wl=[665, 560, 490]).plot.imshow(rgb='wl', robust=True, ax=axs[0, 0])
axs[0, 0].set_title(f'True colour, {date}')
axs[0, 0].set_aspect('equal')
axs[0, 0].set_axis_off()
for ax, (var, kwargs) in zip(axs.ravel()[1:], maps):
    da = l2b[var].isel(Nclasses=0) if 'Nclasses' in l2b[var].dims else l2b[var]
    show(ax, da, var, **kwargs)

## Full resolution

The same chain on level `0`, written to a NetCDF file by `run()` (`.zarr` for a Zarr
store). The computation is lazy: data are read and processed chunk by chunk while the
file is written.

In [ ]:
odir = tempfile.mkdtemp(prefix='grsl2bgen_')
l2b_path = os.path.join(odir, 'S2A_MSIL2B_20170412T110621_N0500_R137_T30TWT_20231111T110836_V3.0.1dev.nc')

proc = GRSl2bgen.Process(l2a_path, l2b_path=l2b_path, level=0)
%time proc.run()

with xr.open_dataset(l2b_path) as ds:
    print(f'{os.path.basename(l2b_path)}: {ds.sizes["y"]} x {ds.sizes["x"]} pixels, '
          f'{len(ds.data_vars)} variables, {os.path.getsize(l2b_path) / 1e6:.0f} MB')

## Command line

The same processing from a terminal. Without `-o`, the output is written to `--odir`
(default: current directory) with `L2AGRS` replaced by `L2B` in the input name; add
`--level k` for a coarse level and `--no_clobber` to skip products already processed:

```bash
GRSl2bgen <l2a_path> --odir <output_dir>              # full resolution
GRSl2bgen <l2a_path> --odir <output_dir> --level 2    # quick look
```

Each command runs in its own Python process, which releases the memory between
products: prefer it to a notebook loop for long batches.

Remove the temporary output:

In [ ]:
shutil.rmtree(odir)